# Transformers

In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Embedding, LayerNormalization, MultiHeadAttention
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np


In [3]:
human_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "thank you",
    "goodbye",
    "i love you",
    "banana",
    "what is your name"
]

minion_sentences = [

    "<start> bello <end>",
    "<start> hana dul sae <end>",
    "<start> me fine <end>",
    "<start> tank yu <end>",
    "<start> po ka <end>",
    "<start> me want banana <end>",
    "<start> banana <end>",
    "<start> whooo your namo <end>"

]


In [ ]:


human_tokenizer = Tokenizer(filters='')
human_tokenizer.fit_on_texts(human_sentences)
human_sequences = human_tokenizer.texts_to_sequences(human_sentences)

minion_tokenizer = Tokenizer(filters='')
minion_tokenizer.fit_on_texts(minion_sentences)
minion_sequences = minion_tokenizer.texts_to_sequences(minion_sentences)





17
[[1, 5, 2], [1, 6, 7, 8, 2], [1, 3, 9, 2], [1, 10, 11, 2], [1, 12, 13, 2], [1, 3, 14, 4, 2], [1, 4, 2], [1, 15, 16, 17, 2]]


In [ ]:
max_len = 10
encoder_input = pad_sequences(human_sequences, maxlen=max_len,padding='post')
print(encoder_input)
decoder_input = pad_sequences(minion_sequences,maxlen=max_len,padding='post')
print(decoder_input)

[[ 3  0  0  0  0  0  0  0  0  0]
 [ 4  5  1  0  0  0  0  0  0  0]
 [ 2  6  7  0  0  0  0  0  0  0]
 [ 8  1  0  0  0  0  0  0  0  0]
 [ 9  0  0  0  0  0  0  0  0  0]
 [ 2 10  1  0  0  0  0  0  0  0]
 [11  0  0  0  0  0  0  0  0  0]
 [12 13 14 15  0  0  0  0  0  0]]
[[ 1  5  2  0  0  0  0  0  0  0]
 [ 1  6  7  8  2  0  0  0  0  0]
 [ 1  3  9  2  0  0  0  0  0  0]
 [ 1 10 11  2  0  0  0  0  0  0]
 [ 1 12 13  2  0  0  0  0  0  0]
 [ 1  3 14  4  2  0  0  0  0  0]
 [ 1  4  2  0  0  0  0  0  0  0]
 [ 1 15 16 17  2  0  0  0  0  0]]


In [ ]:
decoder_target = np.zeros_like(decoder_input)

decoder_target[:, :-1] = decoder_input[:, 1:]

print(decoder_target)

[[ 5  2  0  0  0  0  0  0  0  0]
 [ 6  7  8  2  0  0  0  0  0  0]
 [ 3  9  2  0  0  0  0  0  0  0]
 [10 11  2  0  0  0  0  0  0  0]
 [12 13  2  0  0  0  0  0  0  0]
 [ 3 14  4  2  0  0  0  0  0  0]
 [ 4  2  0  0  0  0  0  0  0  0]
 [15 16 17  2  0  0  0  0  0  0]]


In [ ]:
human_vocab_size = len(human_tokenizer.word_index) + 1

minion_vocab_size = len(minion_tokenizer.word_index) + 1


print(human_vocab_size)
print(minion_vocab_size)

16
18


In [ ]:
def positional_encoding(max_len, d_model):

    pos = np.arange(max_len)[:, np.newaxis]

    i = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(10000,(2 * (i // 2)) / np.float32(d_model))

    angle_rads = pos * angle_rates

    # sin for even index
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

    # cos for odd index
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(angle_rads,dtype=tf.float32)



In [ ]:
class FeedForward(Layer):
    def __init__(self, d_model, dff):
        super().__init__()

        self.dense1 = Dense(dff,activation='relu')
        self.dense2 = Dense(d_model)

    def call(self, x):
        x = self.dense1(x)
        x = self.dense2(x)
        return x

In [ ]:
class EncoderLayer(Layer):
    def __init__(self,d_model,num_heads,dff):
        super().__init__()

        self.mha = MultiHeadAttention(num_heads=num_heads,key_dim=d_model)
        self.ffn = FeedForward(d_model,dff)
        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

    def call(self, x):

        attn_output = self.mha(x, x, x)
        x = self.norm1(x + attn_output)
        ffn_output = self.ffn(x)
        x = self.norm2(x + ffn_output)
        return x

In [ ]:
class Encoder(Layer):
    def __init__(self,num_layers,vocab_size,d_model,num_heads,dff,max_len):
        super().__init__()

        self.embedding = Embedding(vocab_size,d_model)
        self.pos_encoding = positional_encoding(max_len,d_model)
        self.layers = [ EncoderLayer(d_model, num_heads,dff) for _ in range(num_layers)]

    def call(self, x):
        x = self.embedding(x)
        x = x + self.pos_encoding[:tf.shape(x)[1], :]
        for layer in self.layers:
            x = layer(x)
        return x


In [ ]:
class DecoderLayer(Layer):
    def __init__(self,d_model,num_heads,dff):
        super().__init__()

        self.mha1 = MultiHeadAttention(num_heads=num_heads,key_dim=d_model)
        self.mha2 = MultiHeadAttention(num_heads=num_heads,key_dim=d_model)
        self.ffn = FeedForward(d_model,dff)
        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()
        self.norm3 = LayerNormalization()

    def call(self, x, enc_output):

        attn1 = self.mha1(x,x,x,use_causal_mask=True)
        x = self.norm1(x + attn1)
        attn2 = self.mha2(x,enc_output,enc_output)
        x = self.norm2(x + attn2)
        ffn_output = self.ffn(x)
        x = self.norm3(x + ffn_output)
        return x

In [ ]:
class Decoder(Layer):
    def __init__(self,num_layers,vocab_size,d_model, num_heads,dff,max_len):
        super().__init__()
        self.embedding = Embedding(vocab_size,d_model)
        self.pos_encoding = positional_encoding(max_len,d_model)
        self.layers = [DecoderLayer(d_model,num_heads,dff)for _ in range(num_layers)]

    def call(self,x,enc_output):
        x = self.embedding(x)
        x = x + self.pos_encoding[:tf.shape(x)[1], :]
        for layer in self.layers:
          x = layer(x,enc_output)
        return x

In [ ]:
class Transformer(tf.keras.Model):
    def __init__(self,num_layers,human_vocab_size,minion_vocab_size,d_model,num_heads,dff,max_len):
        super().__init__()
        self.encoder = Encoder(num_layers,human_vocab_size,d_model,num_heads,dff,max_len)
        self.decoder = Decoder(num_layers,minion_vocab_size,d_model,num_heads,dff,max_len)

        self.final_layer = Dense(minion_vocab_size)

    def call(self, inputs):
        enc_input, dec_input = inputs
        enc_output = self.encoder(enc_input)
        dec_output = self.decoder(dec_input,enc_output)
        output = self.final_layer(dec_output)
        return output

In [ ]:
model = Transformer(num_layers=2,human_vocab_size=human_vocab_size,minion_vocab_size=minion_vocab_size,d_model=64,num_heads=4,dff=128,max_len=max_len)

In [ ]:
model.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])


In [ ]:
model.fit([encoder_input, decoder_input],decoder_target,epochs=500,verbose=1)

Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 26s 26s/step - accuracy: 0.0125 - loss: 3.4304
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.6875 - loss: 1.4097
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.6875 - loss: 1.2117
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.7250 - loss: 1.1646
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.7250 - loss: 1.1717
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.7500 - loss: 1.0645
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.6875 - loss: 1.0203
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.6875 - loss: 1.0035
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.6875 - loss: 0.9570
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.7125 - loss: 0.9120
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.7250 - loss: 0.8971
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.7250 - l

In [ ]:
# =========================================================
# 17. TRANSLATION FUNCTION
# =========================================================

index_word = minion_tokenizer.index_word

start_token = minion_tokenizer.word_index['<start>']

end_token = minion_tokenizer.word_index['<end>']

def translate(sentence):

    # Convert sentence to sequence
    seq = human_tokenizer.texts_to_sequences(
        [sentence]
    )

    # Padding
    seq = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post'
    )

    # Decoder input
    decoder_input_test = np.zeros(
        (1, max_len)
    )

    # Put start token
    decoder_input_test[0, 0] = start_token

    output_sentence = []

    # Predict word by word
    for i in range(1, max_len):

        prediction = model.predict(
            [seq, decoder_input_test],
            verbose=0
        )

        predicted_id = np.argmax(
            prediction[0, i - 1]
        )

        # Stop at end token
        if predicted_id == end_token:

            break

        # Convert id → word
        predicted_word = index_word.get(
            predicted_id,
            ''
        )

        # Add word
        output_sentence.append(
            predicted_word
        )

        # Feed word back into decoder
        decoder_input_test[0, i] = predicted_id

    return " ".join(output_sentence)

# =========================================================
# 18. TEST TRANSLATION
# =========================================================

print("\n")

print("hello --->",
      translate("hello"))

print("thank you --->",
      translate("thank you"))

print("banana --->",
      translate("banana"))

print("i love you --->",
      translate("i love you"))

print("goodbye --->",
      translate("goodbye"))



hello ---> bello
thank you ---> tank yu
banana ---> banana
i love you ---> me want banana
goodbye ---> po ka


In [ ]:
a = input("Enter a sentence: ")
print(translate(a))

Enter a sentence: thak you
tank yu
